# NB3 — Traza didáctica del grafo LangGraph (ciclo a ciclo)

Notebook pedagógico para la defensa de tesis: toma una fila del CSV de auditoría
y **reproduce exactamente qué nodo del `StateGraph` se activaría en ese ciclo**,
con qué estado de entrada y qué estado de salida produciría.

**No requiere AirSim, ni GPU, ni conexión de red** — los nodos de hardware se reemplazan
con mocks que leen del CSV registrado.

## Concepto central

```
Una fila del CSV = un ciclo completo del grafo.

El grafo ejecuta:
  capture → [degraded_hover | perception] → policy_router → nodo_activo → motor → END

El notebook «replay» ese ciclo con datos reales en lugar de datos de AirSim en vivo.
```

**Cómo usar:**
1. Ajustar `CSV_PATH` en §1 para apuntar a un `.csv` de producción
2. Ajustar `CYCLE` (o `PRESET`) para seleccionar el ciclo de interés
3. Kernel → Run All Cells

## §0 — Importar el grafo real (en modo dry-run)

In [ ]:
import sys
import json
import math
from pathlib import Path
from typing import Any, Dict, Optional

import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Añadir src/ al path para importar el grafo de producción
SRC_DIR = str(Path('src').resolve())
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
print(f'sys.path[0] = {sys.path[0]}')

In [ ]:
# Importar el grafo de producción y el DroneState
from src.agents.graph import build_workflow, DroneState
from src.perception.obstacle_field import ObstacleField

print('Importación del grafo: OK')
print(f'DroneState keys: {list(DroneState.__annotations__.keys())}')

In [ ]:
# ─── Monkey-patching de nodos que necesitan hardware real ────────────────────
# Se aplica ANTES de invocar build_workflow para que el grafo use los mocks.

import src.agents.graph as _graph_module

_CSV_ROW = {}  # se llenará en §1 con la fila seleccionada


def _mock_capture_node(state: DroneState) -> DroneState:
    """Reemplaza capture_node: devuelve telemetría del CSV, sin llamar a AirSim."""
    row = _CSV_ROW
    obs_field = _parse_obstacle_field(row)
    return {
        **state,
        'rgb_image':    None,
        'prev_image':   None,
        'frame_history': [None],
        'frame_history_ts': [float(row.get('t', 0.))],
        'telemetry': {
            'x': float(row.get('pos_x', 0.)),
            'y': float(row.get('pos_y', 0.)),
            'z': float(row.get('pos_z', 0.)),
            'vx': float(row.get('vel_x', 0.)),
            'vy': float(row.get('vel_y', 0.)),
            'vz': float(row.get('vel_z', 0.)),
            'yaw_deg': float(row.get('yaw_deg', 0.)),
            'pitch_deg': float(row.get('pitch_deg', 0.)),
            'roll_deg': float(row.get('roll_deg', 0.)),
            'has_collided': str(row.get('has_collided', 'False')).lower() == 'true',
            'collision_object': str(row.get('collision_object', '')),
        },
        'obstacle_field': obs_field,
        'degraded': str(row.get('degraded', 'False')).lower() == 'true',
        'wp_index':    int(float(row.get('wp_index', 0))),
        'dist_to_wp_m': float(row.get('dist_to_wp_m', 0.)),
    }


def _mock_motor_node(state: DroneState) -> DroneState:
    """Reemplaza motor_node: imprime el comando resultante sin llamar a AirSim."""
    action = state.get('action', 'N/A')
    vel = state.get('velocity_command', {})
    print(f'  [MOCK motor_node] execute_velocity({vel})  →  action={action}')
    return state


# Parchear los nodos en el módulo antes de build_workflow
if hasattr(_graph_module, 'capture_node'):
    _graph_module.capture_node = _mock_capture_node
    print('  ✅ capture_node parcheado')
if hasattr(_graph_module, 'motor_node'):
    _graph_module.motor_node   = _mock_motor_node
    print('  ✅ motor_node parcheado')

print('Mocks aplicados.')

## §1 — Cargar el CSV y seleccionar una fila

In [ ]:
# ── CAMBIAR AQUÍ ──────────────────────────────────────────────────────────────
CSV_PATH = '../airsim-runs/produccion/tier1/townsim_clear/slm/deep_vlm/seed_1.csv'
CYCLE    = 50   # índice de la fila (0-based). Ver §8 para presets por tipo de ciclo.
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(CSV_PATH)
print(f'CSV cargado: {len(df)} ciclos  |  scenario={df["scenario"].iloc[0]}  arm={df["arm"].iloc[0]}  seed={int(df["seed"].iloc[0])}')

row = df.iloc[CYCLE]
print(f'\nCiclo seleccionado: índice={CYCLE}')
print(f'  cycle={int(row["cycle"])}  |  route={row["route"]}  |  action={row["action"]}')
print(f'  dist_to_wp_m={row["dist_to_wp_m"]:.1f}  |  arm={row["arm"]}')
print(f'  centro_blocked={row["field_centro_blocked"]}  |  centro_ttc_s={row["field_centro_ttc_s"]}')
print(f'  slm_invoked={row["slm_invoked"]}  |  slm_adherent={row["slm_adherent"]}')

## §2 — Reconstruir el DroneState desde la fila CSV

In [ ]:
def _parse_obstacle_field(row: dict) -> ObstacleField:
    """Reconstruye un ObstacleField desde las columnas field_* del CSV."""
    sectors = {}
    for s in ('izquierda', 'centro', 'derecha'):
        occ  = float(row.get(f'field_{s}_occ', 0.) or 0.)
        ttc  = row.get(f'field_{s}_ttc_s')
        ttc  = float(ttc) if ttc not in (None, '', 'nan', float('nan')) else None
        conf = float(row.get(f'field_{s}_conf', 0.) or 0.)
        blk  = str(row.get(f'field_{s}_blocked', 'False')).lower() == 'true'
        sectors[s] = {
            'occupancy': occ, 'ttc_s': ttc,
            'confidence': conf, 'blocked': blk,
        }
    return ObstacleField(sectors=sectors)


def row_to_initial_state(row: pd.Series, arm: str) -> DroneState:
    """Construye el DroneState de entrada para el ciclo a reproducir."""
    row_d = row.to_dict()
    obs_field = _parse_obstacle_field(row_d)
    return DroneState(
        rgb_image=None,
        prev_image=None,
        frame_history=[None],
        frame_history_ts=[float(row_d.get('t', 0.))],
        telemetry={
            'x': float(row_d.get('pos_x', 0.)),
            'y': float(row_d.get('pos_y', 0.)),
            'z': float(row_d.get('pos_z', 0.)),
            'vx': float(row_d.get('vel_x', 0.)),
            'vy': float(row_d.get('vel_y', 0.)),
            'vz': float(row_d.get('vel_z', 0.)),
            'yaw_deg': float(row_d.get('yaw_deg', 0.)),
            'pitch_deg': float(row_d.get('pitch_deg', 0.)),
            'roll_deg': float(row_d.get('roll_deg', 0.)),
            'has_collided': str(row_d.get('has_collided', 'False')).lower() == 'true',
            'collision_object': str(row_d.get('collision_object', '')),
        },
        obstacle_field=obs_field,
        degraded=str(row_d.get('degraded', 'False')).lower() == 'true',
        route=str(row_d.get('route', '')),
        wp_index=int(float(row_d.get('wp_index', 0))),
        dist_to_wp_m=float(row_d.get('dist_to_wp_m', 0.)),
        action=None,
        velocity_command={},
        flight_status='active',
        deliberations=[],
        last_deliberation=None,
        slm_request_id=None,
    )


# Cargar el estado inicial desde la fila seleccionada
_CSV_ROW.update(row.to_dict())  # actualizar el mock para capture_node
arm_selected = str(row['arm'])
initial_state = row_to_initial_state(row, arm_selected)

obs = initial_state['obstacle_field']
print(f'Estado inicial construido para arm={arm_selected}')
print(f'  ObstacleField.centro: occ={obs.sectors["centro"]["occupancy"]:.2f}  '
      f'ttc_s={obs.sectors["centro"]["ttc_s"]}  '
      f'blocked={obs.sectors["centro"]["blocked"]}')

## §3 — Diagrama del grafo (arquitectura StateGraph)

In [ ]:
diagram = """
graph TD
    capture["capture_node\\n(telemetría + frame)"] --> degraded_router{degraded?}
    degraded_router -->|sí| degraded_hover["degraded_hover\\n(flotar, sin control)"] --> capture
    degraded_router -->|no| perception["perception_node\\n(FlowTTC → ObstacleField)"]
    perception --> policy_router{"policy_router\\n(arm + estado)"}
    policy_router -->|reactive| keep_going["keep_going\\n(guiar a waypoint)"]
    policy_router -->|slm libre| deliberative["deliberative_node\\n(consultar VLM)"]
    policy_router -->|evasiva activa| evasive["evasive_node\\n(ejecutar maniobra)"]
    policy_router -->|girar / deadlock| girar_90["girar_90_node\\n(escaneo profundo)"]
    policy_router -->|fsm| fsm["fsm_node\\n(máquina de estados)"]
    keep_going --> motor
    deliberative --> motor
    evasive --> motor
    girar_90 --> motor
    fsm --> motor
    motor["motor_node\\n(execute_velocity)"] --> END
"""

display(Markdown(f'```mermaid{diagram}```'))

## §4 — Traza del ciclo seleccionado (paso a paso)

Se instrumenta cada nodo para imprimir su entrada y los campos que modifica en el estado.

In [ ]:
execution_trace = []

RELEVANT_KEYS = {
    'capture_node':     ['telemetry', 'obstacle_field', 'degraded', 'wp_index', 'dist_to_wp_m'],
    'perception_node':  ['obstacle_field', 'estimated_ttc'],
    'policy_router':    ['route', 'arm'],
    'keep_going':       ['action', 'velocity_command', 'route'],
    'deliberative_node':['action', 'scene_summary', 'next_action', 'last_deliberation', 'route'],
    'evasive_node':     ['action', 'velocity_command', 'route'],
    'girar_90_node':    ['action', 'velocity_command', 'route'],
    'fsm_node':         ['action', 'velocity_command', 'route'],
    'motor_node':       ['velocity_command', 'action'],
}


def _fmt_val(val) -> str:
    """Formatea un valor de estado para display."""
    if hasattr(val, 'sectors'):
        c = val.sectors.get('centro', {})
        return (f'ObstacleField(centro: occ={c.get("occupancy",0):.2f} '
                f'ttc={c.get("ttc_s")} blk={c.get("blocked")})')
    if isinstance(val, dict) and 'x' in val:
        return f'tel(x={val["x"]:.1f} y={val["y"]:.1f} yaw={val.get("yaw_deg",0):.1f}°)'
    s = str(val)
    return s[:80] + '...' if len(s) > 80 else s


def make_traced_node(name, fn):
    """Envuelve un nodo del grafo con logging de entrada/salida."""
    def traced(state):
        print(f'\n{'─'*60}')
        print(f'NODO: {name}')
        keys_in = RELEVANT_KEYS.get(name, [])
        print('  Entrada relevante:')
        for k in keys_in:
            v = state.get(k)
            print(f'    {k:20s} = {_fmt_val(v)}')
        out = fn(state)
        if out is not None:
            print('  Salida (campos modificados):')
            for k in keys_in:
                v_before = state.get(k)
                v_after  = out.get(k) if isinstance(out, dict) else None
                if v_after is not None and str(v_after) != str(v_before):
                    print(f'    {k:20s} = {_fmt_val(v_after)}  (antes: {_fmt_val(v_before)})')
            execution_trace.append({'node': name, 'in': dict(state), 'out': dict(out)})
        return out
    return traced


# Parchear los nodos del módulo con las versiones tracificadas
NODES_TO_TRACE = ['capture_node', 'perception_node', 'keep_going', 'deliberative_node',
                  'evasive_node', 'girar_90_node', 'fsm_node', 'motor_node']
for node_name in NODES_TO_TRACE:
    fn = getattr(_graph_module, node_name, None)
    if fn is not None:
        setattr(_graph_module, node_name, make_traced_node(node_name, fn))
print('Nodos instrumentados para traza.')

In [ ]:
# Construir el grafo y ejecutar el ciclo seleccionado
import os
os.environ.setdefault('AGENT_ARM', arm_selected)

workflow = build_workflow(arm=arm_selected)

print(f'\n{'═'*60}')
print(f'Ejecutando ciclo {int(row["cycle"])}  (arm={arm_selected}, índice CSV={CYCLE})')
print(f'{'═'*60}')
result = workflow.invoke(initial_state)

## §5 — Análisis de la decisión tomada por el policy_router

In [ ]:
def print_decision_tree(state: DroneState, arm: str) -> None:
    """Muestra cada condición del policy_router y su valor para este ciclo."""
    obs = state.get('obstacle_field')
    centro = obs.sectors.get('centro', {}) if obs else {}
    route_prev = state.get('route', '')
    dist = state.get('dist_to_wp_m', 0.)

    print(f'  arm                   = {arm}')
    print(f'  route anterior        = {route_prev}')
    print(f'  centro.blocked        = {centro.get("blocked")}')
    print(f'  centro.ttc_s          = {centro.get("ttc_s")}')
    print(f'  dist_to_wp_m          = {dist:.1f} m')
    print(f'  degraded              = {state.get("degraded")}')

    print('\n  Reglas del router:')
    if state.get('degraded'):
        print('    → degraded_hover  (AirSim no respondió este ciclo)')
    elif arm == 'reactive':
        print('    → keep_going  (brazo reactive: sin deliberación)')
    elif arm == 'fsm':
        print('    → fsm_node  (brazo fsm: máquina de estados determinista)')
    elif arm == 'slm':
        if route_prev == 'girar_90':
            print('    → girar_90  (evasión activa de deadlock en curso)')
        elif route_prev == 'evasive' and centro.get('blocked'):
            print('    → evasive  (maniobra evasiva comprometida, centro sigue bloqueado)')
        elif centro.get('blocked'):
            print('    → deliberative  (centro bloqueado → consultar VLM)')
        else:
            print('    → keep_going  (sin obstáculos activos → avanzar al waypoint)')


print_decision_tree(initial_state, arm_selected)

## §6 — Verificación: ¿el notebook reproduce la misma route que el CSV registró?

In [ ]:
reproduced_route = result.get('route', 'N/A') if result else 'N/A'
expected_route   = str(row['route'])
match = reproduced_route == expected_route

print(f'Route reproducida : {reproduced_route}')
print(f'Route registrada  : {expected_route}')
print()
if match:
    print('✅ Match — el notebook reprodujo exactamente la decisión de producción.')
else:
    print('⚠️  Divergencia — revisar mock de estado o condiciones del router.')
    print('  Posibles causas:')
    print('  - El estado del ciclo anterior influye en el router (route_prev) y el')
    print('    CSV almacena el state DESPUÉS del ciclo, no ANTES.')
    print('  - El DeliberationService tiene estado asíncrono (resultado del ciclo anterior).')

## §7 — Recorrido de múltiples ciclos consecutivos (ventana)

In [ ]:
# ── CAMBIAR AQUÍ ──────────────────────────────────────────────────────────────
WINDOW_START = max(0, CYCLE - 10)
WINDOW_END   = min(len(df) - 1, CYCLE + 10)
# ─────────────────────────────────────────────────────────────────────────────

print(f'Ciclos {WINDOW_START}–{WINDOW_END}:\n')
print(f'  {"idx":>4}  {"cycle":>5}  {"route":<15}  {"action":<22}  {"centro_blk":<12}  {"ttc":>6}  match')
print('  ' + '─' * 80)

# Versión silenciosa del grafo (sin prints de traza)
import src.agents.graph as _gm2
for node_name in NODES_TO_TRACE:
    fn_original = getattr(_gm2, f'_orig_{node_name}', None)
    # Restaurar los originales si se guardaron, o simplemente usar la versión mock

for idx in range(WINDOW_START, WINDOW_END + 1):
    row_i = df.iloc[idx]
    _CSV_ROW.clear()
    _CSV_ROW.update(row_i.to_dict())
    state_i = row_to_initial_state(row_i, str(row_i['arm']))

    try:
        # Invocar sin prints (stdout suprimido via workaround simple)
        import io, contextlib
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            result_i = workflow.invoke(state_i)
        rep_route = result_i.get('route', '?') if result_i else '?'
        exp_route = str(row_i['route'])
        ok = '✅' if rep_route == exp_route else '⚠️'
    except Exception as exc:
        rep_route = f'ERROR: {exc}'[:30]
        exp_route = str(row_i['route'])
        ok = '❌'

    centro_blk = str(row_i['field_centro_blocked'])
    ttc = str(row_i['field_centro_ttc_s'])[:6]
    print(f'  {idx:>4}  {int(row_i["cycle"]):>5}  {rep_route:<15}  {str(row_i["action"]):<22}  '
          f'{centro_blk:<12}  {ttc:>6}  {ok}')

## §8 — Casos de uso pedagógicos preconfigurados

Cambiar `PRESET` para cargar automáticamente un ciclo representativo del run cargado en §1.

In [ ]:
# ── CAMBIAR AQUÍ ──────────────────────────────────────────────────────────────
PRESET = 'deliberativo'
# Opciones: 'libre' | 'deliberativo' | 'evasivo' | 'deadlock' | 'deep_vlm'
# ─────────────────────────────────────────────────────────────────────────────

def find_preset_cycle(df: pd.DataFrame, preset: str) -> int:
    """Busca el primer ciclo del CSV que corresponde al preset pedido."""
    if preset == 'libre':
        mask = (df['route'].isin(['reactive', 'keep_going'])) & \
               (df['field_centro_blocked'].astype(str).str.lower() == 'false')
    elif preset == 'deliberativo':
        mask = df['slm_invoked'].astype(str).str.lower() == 'true'
    elif preset == 'evasivo':
        mask = df['route'] == 'evasive'
    elif preset == 'deadlock':
        mask = df['route'] == 'girar_90'
    elif preset == 'deep_vlm':
        # Ciclo inmediatamente posterior al inicio de un girar_90
        girar = (df['route'] == 'girar_90')
        transitions = girar & ~girar.shift(1, fill_value=False)
        mask = transitions
    else:
        mask = pd.Series([False] * len(df))

    idx = df.index[mask].tolist()
    if idx:
        return idx[0]
    return 0


preset_idx = find_preset_cycle(df, PRESET)
preset_row = df.iloc[preset_idx]
print(f'PRESET="{PRESET}"  →  índice CSV={preset_idx}  cycle={int(preset_row["cycle"])}')
print(f'  route={preset_row["route"]}  action={preset_row["action"]}')
print(f'  centro_blocked={preset_row["field_centro_blocked"]}  slm_invoked={preset_row["slm_invoked"]}')
print()
print('Para trazar este ciclo, copiar el índice de arriba a CYCLE en §1 y re-ejecutar.')